# Driscoll-Kraay Pipeline - Conservative SE Workflow

This notebook mirrors `02_modern_pipeline.ipynb` but refits every regression
with **Driscoll-Kraay (1998) kernel standard errors** instead of the
two-way-clustered default.

## Why

The Pesaran (2004) CD test on our baseline residuals rejects the null of
cross-sectional independence for every globalisation index (see
`outputs/tables/residual_cd_test.tex`). Under strong CSD:

- **One-way entity clustering** (literature convention) is anti-conservative
  because it assumes errors are independent across countries.
- **Two-way clustering** (entity + time) handles *weak* CSD but becomes
  inconsistent with only T ~ 40 time periods.
- **Driscoll-Kraay** kernel SEs are nonparametric and robust to arbitrary
  cross-sectional dependence plus serial correlation. They are the
  econometrically defensible default when a CD test rejects independence in
  a macro panel.

## Scope

Every regression in this notebook uses `run_panel_ols(..., cov_type="kernel")`.
Point estimates are identical to the two-way-clustered pipeline — only the
standard errors, t-statistics, p-values, and significance stars change.

## Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
from linearmodels.panel import compare

from analysis.regression_utils import (
    LATEX_LABEL_MAP,
    generate_marginal_effects,
    prepare_regression_data,
    run_panel_ols,
    significance_stars,
)
from clean.panel_utils import add_welfare_regimes, create_lags
from clean.utils import load_config

config = load_config()
master = add_welfare_regimes(
    pd.read_parquet(REPO_ROOT / "data" / "final" / "master_dataset.parquet")
)

indices = config.get("indices", ["KOFGI", "KOFEcGI", "KOFSoGI", "KOFPoGI"])
ctrls = config.get(
    "controls",
    ["ln_gdppc", "inflation_cpi", "deficit", "debt", "ln_population", "dependency_ratio"],
)
dep_var = config.get("dependent_var", "sstran")

DK_TABLES_DIR = REPO_ROOT / "outputs" / "tables" / "dk"
DK_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Master panel: {master.shape[0]} rows, {master['iso3'].nunique()} countries, "
      f"{master['year'].min()}-{master['year'].max()}")
print(f"Driscoll-Kraay tables will be written to: {DK_TABLES_DIR}")

Master panel: 1408 rows, 32 countries, 1980-2023
Driscoll-Kraay tables will be written to: /home/user/economics-of-the-welfare-state/outputs/tables/dk


## Helper: fit PanelOLS with Driscoll-Kraay SEs

Wraps `run_panel_ols(..., cov_type="kernel")` so every call in this notebook
uses DK kernel SEs. The function returns a `PanelResults` object that behaves
like any other `linearmodels` fit — same `.params`, `.std_errors`, `.pvalues`,
`.summary` — so we can feed it straight into `compare(...)` for side-by-side
tables and into `generate_marginal_effects(...)` for regime decomposition.

In [2]:
def fit_dk(ols_data, dep_var, exog_vars):
    """Fit PanelOLS with entity+time FEs and Driscoll-Kraay SEs."""
    return run_panel_ols(ols_data, dep_var, exog_vars, cov_type="kernel")


def _summarise(res, indep_var):
    return {
        "Coef": round(float(res.params[indep_var]), 4),
        "DK SE": round(float(res.std_errors[indep_var]), 4),
        "t-stat": round(float(res.tstats[indep_var]), 2),
        "p-value": round(float(res.pvalues[indep_var]), 4),
        "Stars": significance_stars(res.pvalues[indep_var]),
        "N": int(res.nobs),
    }

## Part 1: Baseline regressions (no interactions)

The core specification:

$$
\text{sstran}_{it} = \alpha_i + \lambda_t + \beta\,\text{KOF}_{i,t-1} + \gamma' X_{i,t-1} + \varepsilon_{it}
$$

refit per globalisation index with Driscoll-Kraay SEs.

In [3]:
baseline_models = {}
baseline_summary_rows = []

for idx_name in indices:
    reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        reg_data, dep_var, indep, lagged_ctrls, interactions=False
    )

    res = fit_dk(ols_data, dep_var, exog)
    baseline_models[idx_name] = res

    row = {"Index": idx_name, **_summarise(res, indep)}
    baseline_summary_rows.append(row)

baseline_summary = pd.DataFrame(baseline_summary_rows)
baseline_summary

,Index,Coef,DK SE,t-stat,p-value,Stars,N
0,KOFGI,0.0882,0.0324,2.72,0.0066,***,996
1,KOFEcGI,-0.0199,0.0204,-0.98,0.3289,,996
2,KOFTrGI,-0.0675,0.0170,-3.96,0.0001,***,996
3,KOFFiGI,0.0169,0.0148,1.14,0.2542,,996
4,KOFSoGI,0.1288,0.0293,4.40,0.0000,***,996
5,KOFIpGI,0.1235,0.0239,5.18,0.0000,***,996
6,KOFInGI,0.0484,0.0174,2.77,0.0057,***,996
7,KOFCuGI,0.0291,0.0126,2.30,0.0218,**,996
8,KOFPoGI,0.0458,0.0180,2.55,0.0110,**,996


In [4]:
# Full linearmodels side-by-side table (all coefficients, not just the main index)
baseline_comparison = compare(baseline_models, stars=True)
print(baseline_comparison)

                                                                                          Model Comparison                                                                                          
                                       KOFGI            KOFEcGI            KOFTrGI            KOFFiGI            KOFSoGI            KOFIpGI            KOFInGI            KOFCuGI            KOFPoGI
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                         sstran             sstran             sstran             sstran             sstran             sstran             sstran             sstran             sstran
Estimator                           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS
No. Observation

In [5]:
# Persist LaTeX
latex_str = baseline_comparison.summary.as_latex()
for old, new in LATEX_LABEL_MAP.items():
    latex_str = latex_str.replace(old, new)
out_path = DK_TABLES_DIR / "baseline_regression_table_dk.tex"
out_path.write_text(latex_str, encoding="utf-8")
print(f"Saved: {out_path}")

Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/baseline_regression_table_dk.tex


## Part 2: Heterogeneity across welfare regimes

Regime-interaction specification with social-democratic as the reference
category. `prepare_regression_data(interactions=True)` handles the
construction of `int_conservative`, `int_mediterranean`, `int_liberal`, and
`int_post_communist` terms.

In [6]:
interaction_models = {}

for idx_name in indices:
    reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
    indep = f"{idx_name}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        reg_data, dep_var, indep, lagged_ctrls, interactions=True
    )
    interaction_models[idx_name] = fit_dk(ols_data, dep_var, exog)

interaction_comparison = compare(interaction_models, stars=True)
print(interaction_comparison)

                                                                                          Model Comparison                                                                                          
                                       KOFGI            KOFEcGI            KOFTrGI            KOFFiGI            KOFSoGI            KOFIpGI            KOFInGI            KOFCuGI            KOFPoGI
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                         sstran             sstran             sstran             sstran             sstran             sstran             sstran             sstran             sstran
Estimator                           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS           PanelOLS
No. Observation

In [7]:
latex_str = interaction_comparison.summary.as_latex()
for old, new in LATEX_LABEL_MAP.items():
    latex_str = latex_str.replace(old, new)
out_path = DK_TABLES_DIR / "interaction_regression_table_dk.tex"
out_path.write_text(latex_str, encoding="utf-8")
print(f"Saved: {out_path}")

Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/interaction_regression_table_dk.tex


### Marginal effects by welfare regime (DK SEs)

`generate_marginal_effects` takes a fitted interaction model and computes
the marginal effect of the lagged globalisation index for each regime, using
linear combinations of the interaction coefficients. Because the fitted
model carries DK standard errors, the derived SEs are DK-consistent too.

In [8]:
for idx_name, res in interaction_models.items():
    g_var = f"{idx_name}_lag1"
    me_df = generate_marginal_effects(res, g_var)
    print(f"\n=== Marginal effects ({idx_name}, DK SEs) ===")
    display(me_df)

    out_path = DK_TABLES_DIR / f"marginal_effects_{idx_name}_dk.tex"
    out_path.write_text(me_df.to_latex(index=False, escape=False), encoding="utf-8")


=== Marginal effects (KOFGI, DK SEs) ===


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.143041,0.039167,-3.652090,0.000275,***
1,Conservative,-0.005526,0.036029,-0.153363,0.878146,
2,Mediterranean,0.050220,0.121580,0.413060,0.679659,
3,Liberal,0.088190,0.053882,1.636738,0.102027,
4,Post-Communist,0.060799,0.039822,1.526772,0.127161,



=== Marginal effects (KOFEcGI, DK SEs) ===


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.186536,0.037923,-4.918815,0.000001,***
1,Conservative,-0.112058,0.031143,-3.598200,0.000337,***
2,Mediterranean,-0.028045,0.090440,-0.310098,0.756557,
3,Liberal,0.012299,0.034350,0.358034,0.720400,
4,Post-Communist,-0.019638,0.016240,-1.209278,0.226867,



=== Marginal effects (KOFTrGI, DK SEs) ===


...

,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.098793,0.030659,-3.222310,0.001316,***
1,Conservative,-0.022223,0.016090,-1.381143,0.167570,
2,Mediterranean,0.048230,0.062476,0.771975,0.440327,
3,Liberal,0.025350,0.021561,1.175757,0.239996,
4,Post-Communist,0.039834,0.033604,1.185415,0.236159,



=== Marginal effects (KOFCuGI, DK SEs) ===


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),0.039814,0.031826,1.250985,0.211258,
1,Conservative,0.086522,0.018323,4.722061,0.000003,***
2,Mediterranean,-0.110196,0.050885,-2.165569,0.030600,**
3,Liberal,0.007653,0.029250,0.261657,0.793644,
4,Post-Communist,0.030345,0.026893,1.128361,0.259461,



=== Marginal effects (KOFPoGI, DK SEs) ===


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.083524,0.020622,-4.050177,0.000055,***
1,Conservative,0.025632,0.026987,0.949807,0.342460,
2,Mediterranean,0.066808,0.103264,0.646959,0.517819,
3,Liberal,0.053351,0.043304,1.232030,0.218252,
4,Post-Communist,0.049911,0.028694,1.739467,0.082287,*


## Part 3: GFC subperiod splits

Refits the baseline spec within two GFC subperiods using DK SEs. This
is where the CSD concern bites hardest — the post-GFC sample is shorter, so
there are fewer time periods over which common shocks can average out.

In [9]:
SUBPERIODS = {
    "pre_gfc": (1980, 2007),
    "post_gfc": (2008, 2023),
}

subperiod_rows = []
subperiod_models_by_era = {}

for era, (start, end) in SUBPERIODS.items():
    models_era = {}
    for idx_name in indices:
        reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
        period_data = reg_data[(reg_data["year"] >= start) & (reg_data["year"] <= end)].copy()
        indep = f"{idx_name}_lag1"
        lagged_ctrls = [f"{v}_lag1" for v in ctrls]
        ols_data, exog = prepare_regression_data(
            period_data, dep_var, indep, lagged_ctrls, interactions=False
        )
        if len(ols_data) < len(exog) + 10:
            continue
        res = fit_dk(ols_data, dep_var, exog)
        models_era[idx_name] = res
        subperiod_rows.append({"Period": era, "Index": idx_name, **_summarise(res, indep)})
    if models_era:
        subperiod_models_by_era[era] = models_era

subperiod_summary = pd.DataFrame(subperiod_rows)
subperiod_summary

,Period,Index,Coef,DK SE,t-stat,p-value,Stars,N
0,pre_gfc,KOFGI,0.0293,0.0224,1.31,0.1922,,487
1,pre_gfc,KOFEcGI,-0.0234,0.0407,-0.58,0.5655,,487
2,pre_gfc,KOFTrGI,-0.0646,0.0380,-1.70,0.0896,*,487
3,pre_gfc,KOFFiGI,0.0094,0.0225,0.42,0.6780,,487
4,pre_gfc,KOFSoGI,0.1066,0.0527,2.02,0.0440,**,487
5,pre_gfc,KOFIpGI,0.1330,0.0409,3.26,0.0012,***,487
6,pre_gfc,KOFInGI,0.0073,0.0214,0.34,0.7321,,487
7,pre_gfc,KOFCuGI,0.0482,0.0206,2.34,0.0199,**,487
8,pre_gfc,KOFPoGI,0.0217,0.0179,1.21,0.2254,,487
9,post_gfc,KOFGI,0.1370,0.0542,2.53,0.0118,**,509


In [10]:
for era, models_era in subperiod_models_by_era.items():
    cmp = compare(models_era, stars=True)
    latex_str = cmp.summary.as_latex()
    for old, new in LATEX_LABEL_MAP.items():
        latex_str = latex_str.replace(old, new)
    out_path = DK_TABLES_DIR / f"baseline_regressions_{era}_dk.tex"
    out_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved: {out_path}")

Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/baseline_regressions_pre_gfc_dk.tex
Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/baseline_regressions_post_gfc_dk.tex


### Marginal effects by welfare regime — GFC subperiods (DK SEs)

Fits the interaction specification on each GFC subperiod and computes
marginal effects per welfare regime. Because the model carries DK kernel
SEs, the linear-combination SEs are DK-consistent.

In [11]:
gfc_interaction_models = {}

for era, (start, end) in SUBPERIODS.items():
    print(f"\n{'='*60}")
    print(f"{era.upper()} ({start}-{end}): Interaction regressions with DK SEs")
    print("=" * 60)
    era_models = {}
    for idx_name in indices:
        reg_data = create_lags(master, [idx_name] + ctrls, lags=[1])
        period_data = reg_data[
            (reg_data["year"] >= start) & (reg_data["year"] <= end)
        ].copy()
        indep = f"{idx_name}_lag1"
        lagged_ctrls = [f"{v}_lag1" for v in ctrls]
        ols_data, exog = prepare_regression_data(
            period_data, dep_var, indep, lagged_ctrls, interactions=True
        )
        if len(ols_data) < len(exog) + 10:
            print(f"  {idx_name}: skipped (too few obs)")
            continue
        res = fit_dk(ols_data, dep_var, exog)
        era_models[idx_name] = res

        g_var = f"{idx_name}_lag1"
        me_df = generate_marginal_effects(res, g_var)
        print(f"\n--- Marginal effects ({idx_name}, DK SEs) ---")
        display(me_df)

        out_path = DK_TABLES_DIR / f"marginal_effects_{idx_name}_{era}_dk.tex"
        out_path.write_text(me_df.to_latex(index=False, escape=False), encoding="utf-8")
        print(f"Saved: {out_path}")

    gfc_interaction_models[era] = era_models


PRE_GFC (1980-2007): Interaction regressions with DK SEs

--- Marginal effects (KOFGI, DK SEs) ---


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.185302,0.038918,-4.761349,0.000003,***
1,Conservative,-0.042655,0.104940,-0.406470,0.684601,
2,Mediterranean,-0.139332,0.078627,-1.772057,0.077096,*
3,Liberal,0.030875,0.079672,0.387529,0.698557,
4,Post-Communist,0.085299,0.036951,2.308429,0.021451,**


Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/marginal_effects_KOFGI_pre_gfc_dk.tex



--- Marginal effects (KOFEcGI, DK SEs) ---


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.180341,0.035327,-5.104918,4.988613e-07,***
1,Conservative,-0.081457,0.032958,-2.471567,1.384040e-02,**
2,Mediterranean,-0.149680,0.081418,-1.838414,6.669418e-02,*
3,Liberal,-0.023425,0.028042,-0.835367,4.039773e-01,
4,Post-Communist,0.041428,0.013138,3.153288,1.728093e-03,***


...

,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.034375,0.013852,-2.481599,0.013443,**
1,Conservative,-0.012980,0.007182,-1.807339,0.071376,*
2,Mediterranean,-0.059790,0.045227,-1.321995,0.186840,
3,Liberal,-0.032091,0.014808,-2.167072,0.030752,**
4,Post-Communist,-0.004146,0.035496,-0.116794,0.907075,


Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/marginal_effects_KOFCuGI_post_gfc_dk.tex



--- Marginal effects (KOFPoGI, DK SEs) ---


,Welfare Regime,Marginal Effect,Std. Error,t-stat,p-value,Sig.
0,Social Democrat (Ref),-0.156100,0.075329,-2.072247,0.038810,**
1,Conservative,0.008854,0.028637,0.309179,0.757328,
2,Mediterranean,-0.241547,0.093216,-2.591246,0.009873,***
3,Liberal,0.000948,0.110992,0.008540,0.993190,
4,Post-Communist,0.062789,0.054624,1.149493,0.250962,


Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/marginal_effects_KOFPoGI_post_gfc_dk.tex


## Part 4: Feedback regressions (reverse causality)

Swaps the dependent and the globalisation variable to test whether welfare
spending predicts future globalisation:

$$
\text{KOF}_{it} = \alpha_i + \lambda_t + \beta\,\text{sstran}_{i,t-1} + \gamma' X_{i,t-1} + \varepsilon_{it}
$$

Same DK SE convention throughout.

In [12]:
feedback_models = {}
feedback_rows = []

for idx_name in indices:
    feedback_ctrls = [dep_var] + ctrls  # sstran is now a regressor
    reg_data = create_lags(master, [idx_name] + feedback_ctrls, lags=[1])
    indep = f"{dep_var}_lag1"
    lagged_ctrls = [f"{v}_lag1" for v in ctrls]
    ols_data, exog = prepare_regression_data(
        reg_data, idx_name, indep, lagged_ctrls, interactions=False
    )
    if len(ols_data) < len(exog) + 10:
        continue
    res = fit_dk(ols_data, idx_name, exog)
    feedback_models[idx_name] = res
    feedback_rows.append({"Dependent": idx_name, **_summarise(res, indep)})

feedback_summary = pd.DataFrame(feedback_rows)
feedback_summary

,Dependent,Coef,DK SE,t-stat,p-value,Stars,N
0,KOFGI,0.1252,0.0446,2.81,0.0051,***,996
1,KOFEcGI,-0.1018,0.0907,-1.12,0.2622,,996
2,KOFTrGI,-0.2554,0.0736,-3.47,0.0005,***,996
3,KOFFiGI,0.0508,0.1258,0.40,0.6862,,996
4,KOFSoGI,0.2249,0.0299,7.52,0.0000,***,996
5,KOFIpGI,0.1965,0.0322,6.11,0.0000,***,996
6,KOFInGI,0.1515,0.0886,1.71,0.0878,*,996
7,KOFCuGI,0.3145,0.0725,4.34,0.0000,***,996
8,KOFPoGI,0.2530,0.0607,4.17,0.0000,***,996


In [13]:
feedback_comparison = compare(feedback_models, stars=True)
latex_str = feedback_comparison.summary.as_latex()
for old, new in LATEX_LABEL_MAP.items():
    latex_str = latex_str.replace(old, new)
out_path = DK_TABLES_DIR / "feedback_regression_table_dk.tex"
out_path.write_text(latex_str, encoding="utf-8")
print(f"Saved: {out_path}")

Saved: /home/user/economics-of-the-welfare-state/outputs/tables/dk/feedback_regression_table_dk.tex


## Summary of the DK pipeline

Because switching the covariance estimator does not move the point
estimates, the headline coefficient on the lagged globalisation index is the
same as in `02_modern_pipeline.ipynb`. What changes:

- **Standard errors** — generally larger under DK than under one-way entity
  clustering, because DK allows arbitrary cross-sectional dependence.
- **p-values and significance stars** — correspondingly more conservative.
  Effects that were significant under one-way entity clustering but lose
  their stars here were being propped up by the (violated) independence
  assumption.
- **Economic story** — unchanged in direction, but the range of robust
  conclusions narrows.

## When to prefer which SE

| Context | Recommended SE |
| --- | --- |
| Reviewer familiarity / historical comparability | One-way entity clustering |
| Project-internal default (current) | Two-way clustering |
| Residual CSD present (our case) | **Driscoll-Kraay** |

For the DK-skeptical reader, the side-by-side `se_comparison.tex` and
`se_comparison_{era}.tex` tables (produced by
`export_subperiod_se_comparison_tables`) show all three columns next to each
other — see `03_se_comparison.ipynb`.